# Setup

### Deps

In [62]:
# Install Deps
%pip install anthropic python-dotenv tree-sitter tree-sitter-c-sharp

Note: you may need to restart the kernel to use updated packages.


### Imports And Client

In [84]:
# Setup
from dotenv import load_dotenv
from anthropic import Anthropic
import json, ast
from tree_sitter import Language, Parser
import tree_sitter_c_sharp as tscsharp
import statistics

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"
msg_queue = []

### Chat Functions

In [85]:
system_prompt = None


def add_msg(queue, msg, role):
    queue.append(
        {
            "role": role,
            "content": msg           
        }
    )

def claude_response(msg_queue, extra_params=None):   
    params = {
        "model": model,
        "max_tokens": 1000,
        "extra_headers": {"anthropic-workspace-id": "wrkspc_01RNMpstasBQ8NVXbRqZoota"},
        "messages": msg_queue,
    }

    if system_prompt:
        params["system"] = system_prompt

    if extra_params:
        params.update(extra_params)

    return client.messages.stream(**params)
     
def chat(queue, msg, extra_params=None, assistant_message=None):
    add_msg(queue, msg, "user")
    if assistant_message:
        add_msg(queue ,assistant_message, "assistant")
    return claude_response(queue, extra_params)

### Prompt Evaluation Functions

In [111]:
def eval_test_case(test_case):
    eval_prompt = f"""
    Please Solve The Following Task:

    {test_case['task']}

    * Only Provide Code, no comments or commentary.
    """
    eval_queue = []
    add_msg(eval_queue, eval_prompt, "user")
    add_msg(eval_queue, "```code", "assistant")
    with claude_response(eval_queue, {"stop_sequences": ["```"]}) as resp:
       output = resp.get_final_message().content[-1].text

    model_grade = grade_by_model({
       "task": test_case['task'],
       "criteria": test_case['criteria'],
       "output": output 
    })

    code_grade = grade_by_code({
        "task": test_case['task'],
        "output": output,
        "language": test_case['language']
    })

    return {
        "test_case": test_case,
        "output": output,
        "strengths": model_grade['strengths'],
        "weaknesses": model_grade['weaknesses'],
        "reasoning": model_grade['reasoning'],
        "model_grade": model_grade['score'],
        "code_grade": code_grade['score'],
        "code_grading_result": code_grade['reason'],
        "score": (model_grade['score'] + code_grade['score']) / 2
    }

def grade_by_model(test_case):
    prompt = f"""
    You are performing prompt evaluation, Please Grade the AI-Generated response to the given task.

    The task:
    <task>
    {test_case['task']}
    </task>

    The Criteria given alongside the task:
    <criteria>
    {test_case['criteria']}
    </criteria>
    
    The Generated Output:
    <output>
    {test_case['output']}
    </output>

    Your Evaluation format:
    Provide your evaluation as structured JSON with the folowing fields:
    - strengths: String array of 1-3 strengths
    - weaknesses: String array of 1-3 weaknesses
    - reasoning: A string of your explanation for the overall score
    - score: a number from 1 to 10, to 2 decimals

    Example JSON Format, which you should stick EXACTLY to:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number,
    }}

    * Do not be overly lenient.
    * Take into account whether or not the model was able to interpret the code language that the task asked for.
    """

    grading_queue = []
    add_msg(grading_queue, prompt, "user")
    add_msg(grading_queue, "```json", "assistant")

    with claude_response(grading_queue, {"stop_sequences": ["```"]}) as output:
        return json.loads(output.get_final_message().content[-1].text)

def grade_by_code(test_case):
    match (test_case['language'].lower()):
        case 'python':
            try:
                ast.parse(test_case['output'].strip())
                return {"score": 10, "reason": "Python Code Parsed Succesfully"}
            except SyntaxError as e:
                return {"score": 0, "reason": f"Encountered Parsing Error: {e}"}

        case 'json':
            try:
                json.loads(test_case['output'].strip())
                return {"score": 10, "reason": "JSON Code Parsed Succesfully"}
            except json.JSONDecodeError as e:
                return {"score": 0, "reason": f"Encountered Parsing Error: {e}"}

        case 'c#':
                parser = Parser(Language(tscsharp.language()))
                tree = parser.parse(test_case['output'].strip().encode())
                if tree.root_node.has_error:
                    return {"score": 0, "reason": "c# code bad"}
                else:
                    return {"score": 10, "reason": "C# Code Parsed Succesfully"}

        case _:
            return {"score": 0, "reason": "invalid language key"}



def run_tests(dataset):
    results = []

    for case in dataset:
        result = eval_test_case(case)
        results.append(result)

    return results

### Generate Test Cases

In [93]:
msg = """
generate for me 10 test cases that will be used for evaluating prompt outputs. This Prompt will be exclusively used for generating different snippets of code in different. The Cases should be in the format of JSON keys and values.:

example Test Case Format:
{
    "task": "Desription of task",
    "language": "The programming language the output code is expected to be written in",
    "criteria": "a string that represents a list of criteria for the model to follow and ensure high quality outputs"
}

* Focus on sticking EXACTLY to the JSON format provided in the example.
* All Test  Cases Must be centered around Python Code Generation.
* Focus that these cases and their criteria are comprehensive and robust enough and have some variance in diffuculty.
* Generate Different Use Cases for the following languages: JSON, Python, C#.
* Order them from easiest to hardest without introucing a separate difficulty score.
"""

with open("test_cases.json", "w") as f:
    with chat(msg_queue, msg, {"stop_sequences": ["```"]}, "```json") as tests:
        json.dump(json.loads(tests.get_final_message().content[-1].text), f, indent=2)

# User Interfaces

### Chat

In [ ]:
msg = "please generate a snippet of python code for looping over something with something similar to `finally` after the loop is done."

with chat(msg_queue, msg, {"stop_sequences": ["```"]}, "```python") as resp:
    for text in resp.text_stream:
        print(text, end="", flush=True)
add_msg(msg_queue, resp.get_final_message().content[-1].text, "assistant")

### Prompt Evaluation

In [112]:
with open("test_cases.json", "r") as f:
    results = run_tests(json.load(f))

with open("test_grades.json", "w") as f:
    json.dump(results, f, indent=2)

model_avg = statistics.mean([result['model_grade']for result in results])
code_avg = statistics.mean([result['code_grade']for result in results])
avg = statistics.mean([result['score']for result in results])

print(f"Model Average is {model_avg}")
print(f"Code Average is {code_avg}")
print(f"Overall Average is {avg}")

Model Average is 5.9
Code Average is 6
Overall Average is 5.95
